# Capítulo 3 — Probabilidade e Estatística

Companion em Python inspirado na sequência conceitual do Volume I de *Market Risk Analysis: Quantitative Methods in Finance*, de Carol Alexander.

> Objetivo: implementar os conceitos matemáticos e financeiros, não reproduzir o texto do livro.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import log, exp, sqrt

from quantfinance.statistics import return_summary

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

In [ ]:
from scipy import stats
from scipy.optimize import minimize

## I.3.2 — Momentos, histogramas, quantis, skewness e kurtosis

In [ ]:
rng = np.random.default_rng(42)
r = stats.t(df=5).rvs(size=5000, random_state=rng) * 0.01

summary = pd.Series(return_summary(r), index=[
    "média", "variância", "volatilidade", "skewness",
    "excesso de kurtosis", "VaR 95% (quantil 5%)"
])
display(summary.to_frame("valor"))

plt.figure(figsize=(8,4))
plt.hist(r, bins=60, density=True)
plt.title("Histograma de retornos Student-t")
plt.show()

## I.3.3 — Distribuições univariadas

In [ ]:
x = np.linspace(-0.06, 0.06, 500)

normal_pdf = stats.norm.pdf(x, loc=0, scale=0.01)
t_pdf = stats.t.pdf(x/0.01, df=5)/0.01

plt.figure(figsize=(8,4))
plt.plot(x, normal_pdf, label="Normal")
plt.plot(x, t_pdf, label="Student t (df=5)")
plt.legend()
plt.title("Normal vs Student-t")
plt.show()

## I.3.3.9–12 — Extremos, Pareto e kernel

In [ ]:
from scipy.stats import genextreme, genpareto, gaussian_kde

sample = stats.t(df=4).rvs(size=3000, random_state=rng)

kde = gaussian_kde(sample)
grid = np.linspace(sample.min(), sample.max(), 500)

plt.figure(figsize=(8,4))
plt.hist(sample, bins=60, density=True, alpha=0.3)
plt.plot(grid, kde(grid))
plt.title("Estimativa kernel da densidade")
plt.show()

threshold = np.quantile(sample, 0.95)
excesses = sample[sample > threshold] - threshold
shape, loc, scale = genpareto.fit(excesses, floc=0)
print("GPD shape:", shape, "scale:", scale)

## I.3.4 — Distribuições multivariadas

In [ ]:
mu = np.array([0.001, 0.0005])
V = np.array([[0.0004, 0.00018],
              [0.00018, 0.0009]])

sample = rng.multivariate_normal(mu, V, size=5000)
df = pd.DataFrame(sample, columns=["R1", "R2"])

display(df.cov())
display(df.corr())

## I.3.5 — Intervalos de confiança e testes de hipótese

In [ ]:
x = rng.normal(0.001, 0.02, 100)
mean = np.mean(x)
se = stats.sem(x)
ci = stats.t.interval(0.95, df=len(x)-1, loc=mean, scale=se)

print("Média:", mean)
print("IC 95%:", ci)

tstat, pvalue = stats.ttest_1samp(x, popmean=0)
print("Teste H0: média = 0")
print("t =", tstat, "p =", pvalue)

## I.3.6 — Máxima verossimilhança

In [ ]:
data = rng.normal(0.002, 0.015, 1000)

def neg_loglike(theta):
    mu, sigma = theta
    if sigma <= 0:
        return np.inf
    return -np.sum(stats.norm.logpdf(data, loc=mu, scale=sigma))

res = minimize(neg_loglike, x0=[0.0, 0.02], method="Nelder-Mead")
print("MLE mu, sigma:", res.x)
print("Comparação com média/std amostral:", data.mean(), data.std(ddof=0))

## I.3.7 — Processos estocásticos, random walk, mean reversion e saltos

In [ ]:
# Ornstein-Uhlenbeck (mean reversion)
theta, mu_ou, sigma_ou = 2.0, 0.0, 0.3
dt = 1/252
n = 1000
x = np.zeros(n)

for t in range(1, n):
    x[t] = x[t-1] + theta*(mu_ou-x[t-1])*dt + sigma_ou*np.sqrt(dt)*rng.normal()

plt.figure(figsize=(9,4))
plt.plot(x)
plt.title("Processo mean-reverting (OU)")
plt.show()

In [ ]:
# Jump-diffusion simples
S0 = 100
mu, sigma = 0.08, 0.2
lam = 1.0
jump_mu, jump_sigma = -0.08, 0.12
steps = 252
dt = 1/steps

S = [S0]
for _ in range(steps):
    z = rng.normal()
    N = rng.poisson(lam*dt)
    jump = np.sum(rng.normal(jump_mu, jump_sigma, N)) if N > 0 else 0.0
    lr = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*z + jump
    S.append(S[-1]*np.exp(lr))

plt.figure(figsize=(9,4))
plt.plot(S)
plt.title("Preço com difusão e saltos")
plt.show()